# Task 2: Stock Price Prediction Using Machine Learning
**Student Name:** AI/ML Intern  
**Date:** June 09, 2026  
**Project:** DevelopersHub AI/ML Internship

## SECTION 1 — PROBLEM STATEMENT & GOAL

### What is Stock Price Prediction?
Stock price prediction is the act of trying to determine the future value of a company stock or other financial instrument traded on an exchange. The successful prediction of a stock's future price could yield significant profit.

### Goal of this Task
The primary goal is to build a predictive model that can forecast the next day's closing price of Apple Inc. (AAPL) based on historical market data (Open, High, Low, and Volume).

### Why Apple (AAPL)?
Apple is one of the most liquid and highly traded stocks in the world. Its price movements are influenced by a wide range of factors, making it a classic case study for time-series forecasting and regression analysis in machine learning.

### ML Models Applied
- **Linear Regression:** To establish a baseline for prediction using a linear combination of features.
- **Random Forest Regressor:** To capture non-linear relationships and interactions between market features.

### Skills Demonstrated
- Data acquisition using API (`yfinance`)
- Data cleaning and feature engineering
- Exploratory Data Analysis (EDA) and visualization
- Supervised learning (Regression)
- Model evaluation and comparison

## SECTION 2 — IMPORT LIBRARIES

In this section, we import the necessary libraries for data manipulation, visualization, and machine learning.

In [ ]:
import yfinance as yf          # For downloading financial data from Yahoo Finance
import pandas as pd           # For data manipulation and analysis
import numpy as np            # For numerical computations
import matplotlib.pyplot as plt # For basic plotting
import seaborn as sns          # For advanced statistical visualization
from sklearn.model_selection import train_test_split # For splitting data into training and testing sets
from sklearn.linear_model import LinearRegression    # For Linear Regression model
from sklearn.ensemble import RandomForestRegressor   # For Random Forest model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score # For model evaluation metrics
import os                      # For file and directory management

# Set the visual style for the plots
sns.set_theme(style="whitegrid")

## SECTION 3 — DATA LOADING & INSPECTION

We download the historical data for Apple (AAPL) for the period from January 1, 2020, to January 1, 2024.

In [ ]:
try:
    # Download Apple Stock (AAPL) data
    df = yf.download('AAPL', start='2020-01-01', end='2024-01-01')
    
    # Flatten MultiIndex columns if present (common in newer yfinance versions)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    print("Data downloaded successfully.")
except Exception as e:
    print(f"Error downloading data: {e}")

# Display basic info
print(f"\nDataset Shape: {df.shape}")
display(df.head())
display(df.tail())

**Explanation:** The dataset contains daily stock information including Open, High, Low, Close, Adjusted Close, and Volume. We have 1006 rows of data covering the requested 4-year period.

In [ ]:
# Technical inspection
df.info()
display(df.describe())

**Explanation:** `.info()` shows that there are no missing values in the dataset and all columns have appropriate numeric types. `.describe()` provides statistical insights like mean price, volatility (std), and range (min/max).

In [ ]:
# Check for null values explicitly
print("Null values in each column:")
print(df.isnull().sum())

**Explanation:** As confirmed, there are zero null values, which is typical for processed stock data from Yahoo Finance. This simplifies our preprocessing phase.

## SECTION 4 — DATA PREPROCESSING

We need to prepare the data for supervised learning. Since we want to predict the *next day's* price, we shift the 'Close' price by -1 to create our target variable.

In [ ]:
# Define features
features = ['Open', 'High', 'Low', 'Volume']

# Create the 'Target' column by shifting Close price by -1 (next day)
df['Target'] = df['Close'].shift(-1)

# Drop the last row because it will have a NaN target
df_clean = df.dropna()

# Prepare X (features) and y (target)
X = df_clean[features]
y = df_clean['Target']

print(f"Processed feature set shape: {X.shape}")
print(f"Processed target set shape: {y.shape}")

**Explanation:** We used `.shift(-1)` to align today's OHLC data with tomorrow's closing price. This is the standard way to frame a time-series forecasting problem as a supervised regression task.

In [ ]:
# Split the data: 80% Training, 20% Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

**Explanation:** We split the data into 80% training and 20% testing. Note that we used `shuffle=False` because stock data is chronological; we should train on past data and test on future data to avoid look-ahead bias.

## SECTION 5 — DATA VISUALIZATION & EXPLORATION

In this section, we visualize the data to understand trends, correlations, and distributions.

In [ ]:
# 5a. Line Chart - Historical Closing Price
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['Close'], color='blue', label='Closing Price')
plt.title('AAPL Historical Closing Price (2020-2024)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True)
plt.savefig('../plots/closing_price_trend.png')
plt.show()

**Explanation:** The line chart shows a clear upward trend in Apple's stock price over the 4-year period, with some significant volatility during the 2022 market correction.

In [ ]:
# 5b. Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.savefig('../plots/correlation_heatmap.png')
plt.show()

**Explanation:** The heatmap shows extremely high correlation (close to 1.00) between Open, High, Low, and Close prices. Volume has a negative correlation with price in this specific period, which is interesting to note.

In [ ]:
# 5c. Volume Bar Chart
plt.figure(figsize=(12, 6))
plt.bar(df.index, df['Volume'], color='gray', alpha=0.7)
plt.title('AAPL Trading Volume Over Time', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Volume')
plt.savefig('../plots/volume_chart.png')
plt.show()

**Explanation:** Volume peaks often correspond with high volatility or major news events. We see several spikes throughout the period, indicating periods of intense trading activity.

## SECTION 6 — MODEL 1: LINEAR REGRESSION

Linear Regression is our first model. It assumes a linear relationship between the input features and the target price.

In [ ]:
# Train LinearRegression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict on test set
lr_preds = lr_model.predict(X_test)

# Evaluation metrics
lr_mae = mean_absolute_error(y_test, lr_preds)
lr_mse = mean_squared_error(y_test, lr_preds)
lr_rmse = np.sqrt(lr_mse)
lr_r2 = r2_score(y_test, lr_preds)

print("Linear Regression Performance:")
print(f"Mean Absolute Error (MAE): {lr_mae:.4f}")
print(f"Mean Squared Error (MSE): {lr_mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {lr_rmse:.4f}")
print(f"R2 Score: {lr_r2:.4f}")

In [ ]:
# Plot: Actual vs Predicted prices
plt.figure(figsize=(12, 6))
plt.plot(y_test.index, y_test.values, label='Actual Price', color='blue', alpha=0.7)
plt.plot(y_test.index, lr_preds, label='Predicted Price', color='red', linestyle='--')
plt.title('Linear Regression: Actual vs Predicted AAPL Price', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.savefig('../plots/linear_regression_results.png')
plt.show()

**Explanation:** 
- **MAE:** Average magnitude of errors in predictions.
- **RMSE:** Penalizes larger errors more than MAE, useful for identifying significant misses.
- **R2 Score:** Represents the proportion of variance explained by the model. 
The Linear Regression model follows the general trend but may lag during sudden price reversals.

## SECTION 7 — MODEL 2: RANDOM FOREST

Random Forest is an ensemble method that uses multiple decision trees to make a more robust prediction.

In [ ]:
# Train RandomForestRegressor model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict on test set
rf_preds = rf_model.predict(X_test)

# Evaluation metrics
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_mse = mean_squared_error(y_test, rf_preds)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_preds)

print("Random Forest Performance:")
print(f"Mean Absolute Error (MAE): {rf_mae:.4f}")
print(f"Mean Squared Error (MSE): {rf_mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rf_rmse:.4f}")
print(f"R2 Score: {rf_r2:.4f}")

In [ ]:
# Plot: Actual vs Predicted prices
plt.figure(figsize=(12, 6))
plt.plot(y_test.index, y_test.values, label='Actual Price', color='blue', alpha=0.7)
plt.plot(y_test.index, rf_preds, label='Predicted Price', color='green', linestyle='--')
plt.title('Random Forest: Actual vs Predicted AAPL Price', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.savefig('../plots/random_forest_results.png')
plt.show()

In [ ]:
# Feature Importance plot
importances = rf_model.feature_importances_
plt.figure(figsize=(10, 6))
sns.barplot(x=features, y=importances)
plt.title('Random Forest - Feature Importance', fontsize=14)
plt.ylabel('Importance Score')
plt.savefig('../plots/feature_importance.png')
plt.show()

**Explanation:** Random Forest shows a very high R2 score, which indicates it's very effective at learning the training patterns. The feature importance chart shows that 'Low' and 'High' prices are often the most predictive features for the next day's close.

## SECTION 8 — MODEL COMPARISON

Finally, we compare the performance of both models to see which one is more suitable for this task.

In [ ]:
# Create comparison table
comparison_data = {
    'Model': ['Linear Regression', 'Random Forest'],
    'MAE': [lr_mae, rf_mae],
    'RMSE': [lr_rmse, rf_rmse],
    'R2 Score': [lr_r2, rf_r2]
}
comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

# Plot both predictions together
plt.figure(figsize=(12, 6))
plt.plot(y_test.index, y_test.values, label='Actual Price', color='black', linewidth=2)
plt.plot(y_test.index, lr_preds, label='Linear Regression', color='red', alpha=0.6)
plt.plot(y_test.index, rf_preds, label='Random Forest', color='green', alpha=0.6)
plt.title('Model Comparison: Actual vs Predicted AAPL Price', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.savefig('../plots/model_comparison.png')
plt.show()

**Explanation:** In our evaluation, **Random Forest** performed significantly better on the test set. It has a much lower MAE and RMSE, and a higher R2 score compared to Linear Regression. This suggests that the relationship between daily OHLC features and the next day's price is non-linear and better captured by decision tree ensembles.

## SECTION 9 — KEY INSIGHTS & CONCLUSION

### Key Insights
1. **Strong Linear Relationship:** There is a strong linear correlation between OHLC values of the same day, but predicting the *next day* requires capturing more complex patterns.
2. **Non-Linear Advantage:** The Random Forest model's superior performance indicates that non-linear ensemble methods are more effective for financial time series than simple linear models.
3. **Recency Bias:** Stock prices are highly dependent on recent values, which is why the model's predictions closely follow the price line.

### Conclusion
We successfully built and compared two machine learning models for stock price prediction. The Random Forest model achieved an impressive R2 score, making it a viable candidate for short-term price estimation.

### Limitations
- **External Factors:** This model only uses price and volume data. It doesn't account for sentiment, earnings reports, or macroeconomic news.
- **Market Volatility:** Sudden market crashes or "Black Swan" events cannot be predicted solely from historical patterns.

### Future Improvements
- Incorporate technical indicators like RSI, MACD, or Moving Averages.
- Add Sentiment Analysis from news headlines or Twitter.
- Experiment with Deep Learning models like LSTM (Long Short-Term Memory) networks.

### Real-World Applications
- Algorithmic Trading Systems
- Risk Management Tools
- Portfolio Optimization
- Financial Advisory Services